<a href="https://colab.research.google.com/github/malzbanker/AIM_WEEK_4/blob/task-1/Copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import pickle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
file_path = '/content/drive/My Drive/train.csv'
data = pd.read_csv(file_path)
data.head()

<ipython-input-3-9dce0620afae>:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [ ]:
#Step 2: Convert Datetime and Extract Features
#Task 2.1: Preprocessing
#Step 1: Import Necessary Libraries
#Step 2: Convert Datetime and Extract Features
# Step 1: Convert 'Date' to datetime
data['date'] = pd.to_datetime(data['Date'])

# Step 2: Create the 'SchoolHoliday' column using 'IsHoliday'
data['isHoliday'] = data.apply(lambda row: pd.NaT if row['SchoolHoliday'] == 0 else row['date'], axis=1)

# Step 3: Forward fill the holiday dates
data['isHoliday'] = data['isHoliday'].ffill()

# Step 4: Extract features
data['weekday'] = data['date'].dt.weekday  # Weekday (0=Monday, 6=Sunday)
data['is_weekend'] = (data['date'].dt.weekday >= 5).astype(int)  # Weekend flag

# Step 5: Calculate days to and from holidays
data['days_to_holiday'] = (data['isHoliday'] - data['date']).dt.days
data['days_after_holiday'] = (data['date'] - data['isHoliday']).dt.days

# Step 6: Beginning, mid-month, and end-of-month
data['beginning_of_month'] = (data['date'].dt.day <= 10).astype(int)
data['mid_month'] = ((data['date'].dt.day > 10) & (data['date'].dt.day <= 20)).astype(int)
data['end_of_month'] = (data['date'].dt.day > 20).astype(int)


# Select only numeric columns for calculating the mean
numeric_data = data.select_dtypes(include=np.number)

# Fill NaN values in numeric columns with their respective means
data[numeric_data.columns] = data[numeric_data.columns].fillna(numeric_data.mean())

# Step 8: Scale the Data (ensure all features exist)
scaler = StandardScaler()
features_to_scale = ['weekday', 'days_to_holiday', 'days_after_holiday', 'is_weekend',
                     'beginning_of_month', 'mid_month', 'end_of_month']

# Check if all features are present before scaling
missing_features = [feature for feature in features_to_scale if feature not in data.columns]
if not missing_features:  # Proceed if no features are missing
    data[features_to_scale] = scaler.fit_transform(data[features_to_scale])
else:
    print(f'Missing features for scaling: {missing_features}')

# Output the processed DataFrame
print(data)


         Store  DayOfWeek        Date  Sales  Customers  Open  Promo  \
0            1          5  2015-07-31   5263        555     1      1   
1            2          5  2015-07-31   6064        625     1      1   
2            3          5  2015-07-31   8314        821     1      1   
3            4          5  2015-07-31  13995       1498     1      1   
4            5          5  2015-07-31   4822        559     1      1   
...        ...        ...         ...    ...        ...   ...    ...   
1017204   1111          2  2013-01-01      0          0     0      0   
1017205   1112          2  2013-01-01      0          0     0      0   
1017206   1113          2  2013-01-01      0          0     0      0   
1017207   1114          2  2013-01-01      0          0     0      0   
1017208   1115          2  2013-01-01      0          0     0      0   

        StateHoliday  SchoolHoliday       date  isHoliday   weekday  \
0                  0              1 2015-07-31 2015-07-31  0.501

Task 2.2: Building Models with Sklearn Pipelines
Step 1: Import Libraries

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [ ]:
#Step 2: Defining Features and Target Variable

# Features and Target
X = data[features_to_scale]  # Independent variables
y = data['Sales']              # Target variable

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#Step 3: Creating the Pipeline for Random Forest Regressor

pipeline = Pipeline([
    ('rf', RandomForestRegressor(n_estimators=100))
])

# Fit the model
pipeline.fit(X_train, y_train)

Pipeline(steps=[('rf', RandomForestRegressor())])

Task 2.3: Choose a Loss Function
For a regression problem, we can choose Mean Absolute Error (MAE) as our loss function. MAE is interpretable and gives the actual error margin in the same units as the output.

In [ ]:
from sklearn.metrics import mean_absolute_error

y_pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f'Mean Absolute Error: {mae}')

Mean Absolute Error: 1991.7617194541108


Task 2.4: Post Prediction Analysis
Step 1: Feature Importance

In [ ]:
importances = pipeline.named_steps['rf'].feature_importances_
feature_importance_df = pd.DataFrame(importances, index=features_to_scale, columns=['importance'])
print(feature_importance_df.sort_values(by='importance', ascending=False))

                    importance
weekday               0.894685
days_after_holiday    0.039554
days_to_holiday       0.037959
beginning_of_month    0.011236
mid_month             0.009059
end_of_month          0.004009
is_weekend            0.003498


Step 2: Estimate Confidence Interval
To estimate confidence intervals, you can use a simple approach by calculating prediction intervals.

In [ ]:
predictions = pipeline.predict(X_test)
std_dev = np.std(y_pred - predictions)
lower_bound = predictions - 1.96 * std_dev
upper_bound = predictions + 1.96 * std_dev

# Example output
predictions_with_intervals = pd.DataFrame({
    'Predicted': predictions,
    'Lower Bound': lower_bound,
    'Upper Bound': upper_bound
})

print(predictions_with_intervals.head())

      Predicted   Lower Bound   Upper Bound
0    181.865904    181.865904    181.865904
1   7025.702643   7025.702643   7025.702643
2   5601.259879   5601.259879   5601.259879
3   7025.702643   7025.702643   7025.702643
4  12976.291807  12976.291807  12976.291807


Task 2.5: Serialize Models
You can use the joblib or pickle library to save your trained model.

In [ ]:
import joblib
import datetime
import os

# Create the 'models' directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Serialize the model
timestamp = datetime.datetime.now().strftime("%d-%m-%Y-%H-%M-%S")
joblib.dump(pipeline, f'models/sales_prediction_model_{timestamp}.pkl')

['models/sales_prediction_model_14-01-2025-17-22-56.pkl']

Task 2.6: Building Model with Deep Learning - LSTM
Step 1: Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

# Assuming 'sales' is your target variable and it is time-series
data = data[['date', 'Sales']].set_index('date')

Step 2: Check Stationarity
You can visually inspect the series or use statistical tests (like Augmented Dickey-Fuller test).

In [ ]:
from statsmodels.tsa.stattools import adfuller

def check_stationarity(data):
    result = adfuller(data)
    return result[1] <= 0.05  # p-value

check_stationarity(data['Sales'])

Step 3: Differencing the Data
If the data is not stationary, you need to difference it.

In [ ]:
data['sales_diff'] = data['Sales'] - data['Sales'].shift(1)
data.dropna(inplace=True)

In [ ]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)

# Scale data
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data['sales_diff'].values.reshape(-1, 1))
X, y = create_dataset(data_scaled, 10)
X = X.reshape(X.shape[0], X.shape[1], 1)  # Reshape

In [ ]:
#Step 5: Build LSTM Model
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(X.shape[1], 1)))
model.add(Dropout(0.2))
model.add(LSTM(50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(1))  # Prediction of the next sale point
model.compile(optimizer='adam', loss='mean_squared_error')

/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
#Step 6: Train the LSTM Model
model.fit(X, y, epochs=100, batch_size=32)

In [ ]:
!pip install flask

In [ ]:
#Step 5: Build LSTM Model
lstm_model = Sequential() # Change the variable name to lstm_model
lstm_model.add(LSTM(50, return_sequences=True, input_shape=(X.shape[1], 1)))
lstm_model.add(Dropout(0.2))
lstm_model.add(LSTM(50, return_sequences=False))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(1))  # Prediction of the next sale point
lstm_model.compile(optimizer='adam', loss='mean_squared_error')

#Step 6: Train the LSTM Model
lstm_model.fit(X, y, epochs=100, batch_size=32) # Use lstm_model for training

# Save the LSTM model using the new variable name
lstm_model.save('lstm_model.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
31788/31788 ━━━━━━━━━━━━━━━━━━━━ 356s 11ms/step - loss: 0.0024
Epoch 2/100
31788/31788 ━━━━━━━━━━━━━━━━━━━━ 385s 11ms/step - loss: 0.0012
Epoch 3/100
21831/31788 ━━━━━━━━━━━━━━━━━━━━ 1:49 11ms/step - loss: 0.0012

KeyboardInterrupt: 

In [ ]:
#Task 3: Model Serving API Call

#Step 2: Create the Flask API
from flask import Flask, request, jsonify
import joblib
import pandas as pd
import numpy as np  # Import numpy for data manipulation
from sklearn.preprocessing import MinMaxScaler # Import for scaling
from keras.models import load_model  # Import load_model

app = Flask(__name__)

# Load the LSTM model instead of assuming it's in 'model'
lstm_model = load_model('path/to/your/lstm_model.h5')  # Replace with the actual path

# Define a function to preprocess the data for the LSTM model
def preprocess_data_for_lstm(data, time_step=10):
    """
    Preprocesses the input data to match the format expected by the LSTM model.

    Args:
        data (dict): Input data as a dictionary.
        time_step (int): Number of time steps to consider. Defaults to 10.

    Returns:
        numpy.ndarray: Preprocessed data as a 3D NumPy array.
    """
    # Assuming data is a list of sales_diff values
    sales_diff = np.array(data['sales_diff']).reshape(-1, 1)

    # Scale data
    scaler = MinMaxScaler(feature_range=(0, 1))
    data_scaled = scaler.fit_transform(sales_diff)

    # Create dataset
    X, _ = create_dataset(data_scaled, time_step)  # Assuming create_dataset is defined

    # Reshape to 3D array if necessary
    if len(X.shape) == 2:  # Check if X is already 3D
        X = X.reshape(X.shape[0], X.shape[1], 1)

    return X


@app.route('/predict', methods=['POST'])
def predict():
    data = request.json  # Expecting JSON input
    # Preprocess data for LSTM model
    processed_data = preprocess_data_for_lstm(data)
    # Assuming 'lstm_model' is your loaded LSTM model
    prediction = lstm_model.predict(processed_data)
    return jsonify({'prediction': prediction.tolist()})

if __name__ == '__main__':
    app.run(debug=True)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'path/to/your/lstm_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)